# 🔮 Тестування BERT-детектора фейкових новин

Цей ноутбук завантажує вже навчену модель і дозволяє перевіряти довільні новини.

In [7]:
import torch
import torch.nn as nn
from transformers import BertTokenizerFast, BertModel
import os
import pickle
import re
from pathlib import Path

import pymorphy3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Пристрій: {device}')

Пристрій: cuda


## 1. Визначення архітектур моделей

In [8]:
class BertClassifierHeadA(nn.Module):
    def __init__(self, bert_model_name, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        hidden = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls_output)
        return self.classifier(x)


class BertClassifierHeadB(nn.Module):
    def __init__(self, bert_model_name, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        hidden = self.bert.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_output)


class AttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, hidden_states, attention_mask):
        scores = self.attention(hidden_states).squeeze(-1)
        scores = scores.masked_fill(attention_mask == 0, float('-inf'))
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (hidden_states * weights).sum(dim=1)


class BertClassifierHeadC(nn.Module):
    def __init__(self, bert_model_name, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        hidden = self.bert.config.hidden_size
        self.attention_pool = AttentionPooling(hidden)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        all_hidden = outputs.last_hidden_state
        pooled = self.attention_pool(all_hidden, attention_mask)
        return self.classifier(pooled)


HEAD_CLASSES = {
    'Head A (Linear)':  BertClassifierHeadA,
    'Head B (BN-MLP)':  BertClassifierHeadB,
    'Head C (AttPool)': BertClassifierHeadC,
}

print('Архітектури визначено.')

Архітектури визначено.


## 2. Завантаження моделі та токенізатора

In [9]:
# Шляхи до файлів (відносно цього ноутбуку)
CHECKPOINT_PATH = 'bert_fake_detector_best.pt'
TOKENIZER_PATH = 'bert_tokenizer/'
THEME_MODEL_PATH = Path('..') / '..' / 'THEME' / 'theme_model.pkl'
THEME_VECTORIZER_PATH = Path('..') / '..' / 'THEME' / 'theme_vectorizer.pkl'
STOPWORDS_PATH = Path('..') / '..' / 'stopwords_ua.txt'

# Завантажуємо чекпоінт
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

head_type = checkpoint['head_type']
model_name = checkpoint['model_name']
MAX_LEN = checkpoint['max_len']

print(f'Тип голови:  {head_type}')
print(f'BERT модель: {model_name}')
print(f'MAX_LEN:     {MAX_LEN}')
print(f'Test Accuracy: {checkpoint["test_accuracy"]:.4f}')
print(f'Test F1:       {checkpoint["test_f1"]:.4f}')

# Ініціалізуємо потрібну архітектуру та завантажуємо ваги
ModelClass = HEAD_CLASSES[head_type]
model = ModelClass(model_name).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Завантажуємо збережений токенізатор
if os.path.isdir(TOKENIZER_PATH):
    tokenizer = BertTokenizerFast.from_pretrained(TOKENIZER_PATH)
    print(f'\nТокенізатор завантажено з {TOKENIZER_PATH}')
else:
    tokenizer = BertTokenizerFast.from_pretrained(model_name)
    print(f'\nТокенізатор завантажено з HuggingFace ({model_name})')

with open(STOPWORDS_PATH, 'r', encoding='utf-8') as file:
    stopwords = set(file.read().split())

morph = pymorphy3.MorphAnalyzer(lang='uk')

with open(THEME_VECTORIZER_PATH, 'rb') as file:
    theme_vectorizer = pickle.load(file)

with open(THEME_MODEL_PATH, 'rb') as file:
    theme_model = pickle.load(file)

print(f'\nМодель тем завантажено з {THEME_MODEL_PATH}')
print('\n✅ Моделі готові до використання!')

C:\Users\igrew\AppData\Local\Temp\ipykernel_23348\2181819793.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(CHECKPOINT_PATH, map_location=device

Тип голови:  Head C (AttPool)
BERT модель: bert-base-multilingual-cased
MAX_LEN:     256
Test Accuracy: 0.9344
Test F1:       0.9344


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Токенізатор завантажено з bert_tokenizer/

Модель тем завантажено з ..\..\THEME\theme_model.pkl

✅ Моделі готові до використання!


## 3. Функція передбачення

In [21]:
WORD_PATTERN = r"[A-Za-zА-Яа-яІіЇїЄєҐґ0-9]+(?:['’`-][A-Za-zА-Яа-яІіЇїЄєҐґ0-9]+)*"


def preprocess_for_theme(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    words = text.split()
    clean_words = [word for word in words if word not in stopwords]
    normalized_words = [morph.parse(word)[0].normal_form for word in clean_words]
    return ' '.join(normalized_words)


def get_fake_probabilities(text: str):
    encoding = tokenizer(
        text,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

    return probs


def explain_fake_prediction(text: str, top_k: int = 5, max_words: int = 24) -> dict:
    """Приблизно оцінює внесок слів у рішення BERT через word occlusion."""
    base_probs = get_fake_probabilities(text)
    predicted_index = int(base_probs.argmax())
    predicted_label = 'TRUE' if predicted_index == 1 else 'FAKE'
    base_probability = float(base_probs[predicted_index])

    unique_words = []
    seen_words = set()
    for word in re.findall(WORD_PATTERN, text.lower()):
        if len(word) < 3 or word.isdigit() or word in seen_words:
            continue
        seen_words.add(word)
        unique_words.append(word)
        if len(unique_words) >= max_words:
            break

    word_impacts = []
    for word in unique_words:
        masked_text = re.sub(rf"(?<!\w){re.escape(word)}(?!\w)", ' ', text, flags=re.IGNORECASE)
        masked_text = re.sub(r'\s+', ' ', masked_text).strip()
        if not masked_text:
            continue

        masked_probs = get_fake_probabilities(masked_text)
        masked_probability = float(masked_probs[predicted_index])
        impact = base_probability - masked_probability
        word_impacts.append({
            'word': word,
            'impact': impact,
        })

    supporting_words = sorted(
        [item for item in word_impacts if item['impact'] > 0],
        key=lambda item: item['impact'],
        reverse=True
    )[:top_k]

    opposing_words = sorted(
        [item for item in word_impacts if item['impact'] < 0],
        key=lambda item: item['impact']
    )[:top_k]

    return {
        'predicted_label': predicted_label,
        'base_probability': base_probability,
        'supporting_words': supporting_words,
        'opposing_words': opposing_words,
        'checked_words': len(word_impacts),
    }


def predict_fake(text: str) -> dict:
    """Повертає передбачення BERT для одного тексту."""
    probs = get_fake_probabilities(text)
    label = 'TRUE ✅' if probs[1] > 0.5 else 'FAKE ❌'
    return {
        'label': label,
        'confidence': float(max(probs)),
        'prob_true': float(probs[1]),
        'prob_fake': float(probs[0]),
    }


def predict_theme(text: str) -> dict:
    """Повертає тему новини через SVM-модель."""
    processed_text = preprocess_for_theme(text)
    theme_vector = theme_vectorizer.transform([processed_text])
    probabilities = theme_model.predict_proba(theme_vector)[0]
    predicted_theme = theme_model.predict(theme_vector)[0]
    all_themes = sorted(
        zip(theme_model.classes_, probabilities),
        key=lambda item: item[1],
        reverse=True
    )

    return {
        'theme': predicted_theme,
        'theme_confidence': float(max(probabilities)),
        'theme_probabilities': all_themes,
    }


def predict(text: str, explain_words: bool = True) -> dict:
    fake_result = predict_fake(text)
    theme_result = predict_theme(text)
    result = {**fake_result, **theme_result}
    if explain_words:
        result['fake_explanation'] = explain_fake_prediction(text)
    return result


def format_probability(value: float) -> str:
    return f'{value * 100:.2f}%'


def show(text: str, explain_words: bool = True, top_k_words: int = 5):
    """Виводить фейковість, тему і приблизний вплив слів у зручному форматі."""
    r = predict(text, explain_words=explain_words)

    print('=' * 80)
    print('АНАЛІЗ НОВИНИ')
    print('=' * 80)
    print(text)
    print()
    print('1. Вердикт щодо правдивості')
    print(f'   Статус: {r["label"]}')
    print(f'   Впевненість моделі: {format_probability(r["confidence"])}')
    print(f'   TRUE: {format_probability(r["prob_true"])}')
    print(f'   FAKE: {format_probability(r["prob_fake"])}')
    print()
    print('2. Тема новини')
    print(f'   Категорія: {r["theme"]}')
    print(f'   Впевненість теми: {format_probability(r["theme_confidence"])}')
    print('   Усі теми:')
    for theme, prob in r['theme_probabilities']:
        print(f'   - {theme}: {format_probability(float(prob))}')

    if explain_words and 'fake_explanation' in r:
        explanation = r['fake_explanation']
        predicted_side = 'TRUE' if r['prob_true'] >= r['prob_fake'] else 'FAKE'
        print()
        print('3. Слова, що вплинули на вердикт')
        print(f'   Оцінка приблизна: прибираємо слово з тексту і дивимось, як змінюється ймовірність класу {predicted_side}.')
        print(f'   Перевірено слів: {explanation["checked_words"]}')

        supporting_words = explanation['supporting_words'][:top_k_words]
        opposing_words = explanation['opposing_words'][:top_k_words]

        if supporting_words:
            print(f'   Найсильніше підштовхують до {predicted_side}:')
            for item in supporting_words:
                print(f'   - {item["word"]}: {item["impact"] * 100:+.2f} п.п.')
        else:
            print(f'   Найсильніше підштовхують до {predicted_side}: явних слів не знайдено.')

        if opposing_words:
            print('   Тягнуть у протилежний бік:')
            for item in opposing_words:
                print(f'   - {item["word"]}: {item["impact"] * 100:+.2f} п.п.')

    print('=' * 80)


print('Функції predict(), explain_fake_prediction() та show() визначено.')

Функції predict(), explain_fake_prediction() та show() визначено.


## 4. Тестування новин

Замінюй тексти нижче на будь-які новини — і запускай клітинку.

In [12]:
news_text = """Встав сюди свою новину"""
show(news_text)

Текст:        Встав сюди свою новину
Передбачення: TRUE ✅
Впевненість:  99.14%
P(TRUE)=99.1%   P(FAKE)=0.9%

Категорія: новини
Впевненість теми: 53.40%

Всі теми:
  бізнес: 0.00%
  новини: 53.40%
  політика: 46.54%
  спорт: 0.06%
  технології: 0.00%


In [24]:
show("Європейський Союз схвалив новий пакет фінансової допомоги Україні для підтримки економіки та відновлення інфраструктури.")

show("Міністерство цифрової трансформації України розширило можливості застосунку Дія, додавши нові електронні документи та послуги.")

show("Українські енергетики відновили електропостачання у кількох населених пунктах після пошкодження енергетичної інфраструктури.")

show("Уряд України оголосив про нову програму підтримки малого та середнього бізнесу через гранти та пільгові кредити.")

show("Українська збірна з футболу оголосила склад команди на найближчі матчі кваліфікації міжнародного турніру.")

show("Національний банк України повідомив про стабільну роботу банківської системи та достатній рівень міжнародних резервів.")

show("У Львові відкрили новий центр реабілітації для військових та цивільних, які постраждали під час війни.")

show("Українські науковці представили нові дослідження у сфері штучного інтелекту на міжнародній науковій конференції.")

show("Міністерство освіти і науки України повідомило про розширення програм академічної мобільності для студентів.")

show("В Україні розпочали новий етап відбудови пошкодженого житла у регіонах, що постраждали від бойових дій.")


АНАЛІЗ НОВИНИ
Європейський Союз схвалив новий пакет фінансової допомоги Україні для підтримки економіки та відновлення інфраструктури.

1. Вердикт щодо правдивості
   Статус: TRUE ✅
   Впевненість моделі: 99.56%
   TRUE: 99.56%
   FAKE: 0.44%

2. Тема новини
   Категорія: бізнес
   Впевненість теми: 63.74%
   Усі теми:
   - бізнес: 63.74%
   - політика: 35.15%
   - спорт: 0.90%
   - новини: 0.14%
   - технології: 0.07%

3. Слова, що вплинули на вердикт
   Оцінка приблизна: прибираємо слово з тексту і дивимось, як змінюється ймовірність класу TRUE.
   Перевірено слів: 13
   Найсильніше підштовхують до TRUE:
   - інфраструктури: +0.81 п.п.
   - допомоги: +0.22 п.п.
   - пакет: +0.22 п.п.
   - україні: +0.14 п.п.
   - новий: +0.08 п.п.
   Тягнуть у протилежний бік:
   - європейський: -0.11 п.п.
   - економіки: -0.10 п.п.
   - союз: -0.09 п.п.
   - схвалив: -0.03 п.п.
   - підтримки: -0.02 п.п.
АНАЛІЗ НОВИНИ
Міністерство цифрової трансформації України розширило можливості застосунку Дія, д